In [ ]:
import scvelo as scv
# scv.logging.print_version()

In [ ]:
scv.settings.verbosity = 3  # show errors(0), warnings(1), info(2), hints(3)
scv.settings.presenter_view = True  # set max width size for presenter view
scv.set_figure_params('scvelo')  # for beautified visualization

# Integrating Loom File and Meta-data

In [ ]:
adata = scv.read_loom("../../2021-01-21_RNA_Velocity_from_2020-12-31_another_version_of_clustering_with_only_E_A_R_samples_re-analyzed_without_IdhR195H/E/velocyto/E.loom")

In [ ]:
X_umap = scv.load('../../2021-01-21_RNA_Velocity_from_2020-12-31_another_version_of_clustering_with_only_E_A_R_samples_re-analyzed_without_IdhR195H/E/2021-01-21_UMAP.csv', index_col = 0)
X_umap

In [ ]:
# clusters = scv.load('../../2021-01-21_RNA_Velocity_from_2020-12-31_another_version_of_clustering_with_only_E_A_R_samples_re-analyzed_without_IdhR195H/E/2021-01-21_clusters.csv', index_col = 0)
clusters = scv.load('../../2021-01-21_RNA_Velocity_from_2020-12-31_another_version_of_clustering_with_only_E_A_R_samples_re-analyzed_without_IdhR195H/E/2023-02-07_clusters.csv', index_col = 0)

clusters

In [ ]:
adata.obs.index

In [ ]:
rename_index = adata.obs.index.str.split(':').str[1]
rename_index = rename_index.str.replace('x', '-1', regex = True)
adata.obs.index = rename_index
adata.obs.index

In [ ]:
adata.obsm['X_umap'] = X_umap.loc[adata.obs_names].values
adata.obs['clusters'] = clusters.loc[adata.obs_names].values
adata.obs['clusters'] = adata.obs['clusters'].astype('category')

In [ ]:
adata

In [ ]:
# adata = adata[adata.obs['clusters'].isin(['stem cells', 'stem cell zone', 'ureter region PC', 
#                                          'lower tubule PC', 'upper tubule PC', 'initial segment PC', 
#                                          'stellate cells', 'Sox21b cells'])].copy()


In [ ]:
adata.obs['clusters'].cat.categories

In [ ]:
# adata.obs['clusters'].cat.reorder_categories(['stem cells', 'stem cell zone', 'ureter region PC', 
#                                          'lower tubule PC', 'upper tubule PC', 'initial segment PC', 
#                                          'stellate cells', 'Sox21b cells'], inplace=True)



In [ ]:
adata.obs['clusters'].cat.categories

In [ ]:
scv.pl.proportions(adata, groupby = 'clusters', dpi = 300)

# Preprocess the Data

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

# Dynamical Model

In [ ]:
#!pip install tqdm
#!pip install ipywidgets

In [ ]:
scv.tl.recover_dynamics(adata, n_jobs=16)

In [ ]:
scv.tl.velocity(adata, mode='dynamical')
scv.tl.velocity_graph(adata)

# Project the velocities

In [ ]:
# scv.pl.velocity_embedding_stream(adata, basis = "umap", color = "clusters",
#                                 palette = ["#80C241","#FFD300","#F58A1F","#EF4E22","#00B6C5","#00A7E0","#8071B3","#002056"],
#                                 dpi = 300, figsize = (7, 7), save = "velocity_embedding_stream.png")


# scv.pl.velocity_embedding_stream(adata, basis = "umap",
#                                  dpi = 300, figsize = (7, 7), save = "velocity_embedding_stream.png")
pal = ["#7ED321","#F5A623","#417505","#F1DF05",
       "#50E3C2","#4A4A4A","#4A90E2","#F199A3",
       "#D0021B","#BD10E0","#BA98FF","#9B9B9B",
       "#000000"]

scv.pl.velocity_embedding_stream(adata,  basis = "umap",color = "clusters",
                                 palette=pal,
                                 dpi = 300, figsize = (7, 7),
                                 save = "velocity_embedding_stream.svg")

In [ ]:
scv.pl.velocity_embedding(adata, arrow_length = 3, arrow_size = 1, 
                          dpi = 300, figsize = (7, 7), 
                          color = "clusters", palette=pal,
                          save = "velocity_embedding.pdf")


# Interprete the velocities

In [ ]:
scv.pl.velocity(adata, ['Dl', 'N'], ncols=1, dpi = 300, save = "velocity.png")

In [ ]:
scv.pl.velocity(adata, ['Dl', 'N'], ncols=2, dpi = 300, save = "velocity_selected.pdf")

# Kinetic rate paramters

In [ ]:
df = adata.var
df = df[(df['fit_likelihood'] > .1) & df['velocity_genes'] == True]

kwargs = dict(xscale='log', fontsize=16)
with scv.GridSpec(ncols=3) as pl:
    pl.hist(df['fit_alpha'], xlabel='transcription rate', **kwargs)
    pl.hist(df['fit_beta'] * df['fit_scaling'], xlabel='splicing rate', xticks=[.1, .4, 1], **kwargs)
    pl.hist(df['fit_gamma'], xlabel='degradation rate', xticks=[.1, .4, 1], **kwargs)

scv.get_df(adata, 'fit*', dropna=True).head()

# Latent time

In [ ]:
scv.tl.latent_time(adata)
scv.pl.scatter(adata, color='latent_time', color_map='gnuplot', size=80)


In [ ]:
top_genes = adata.var['fit_likelihood'].sort_values(ascending=False).index[:30]
scv.pl.heatmap(adata, var_names=top_genes, sortby='latent_time', col_color='clusters', n_convolve=100)


# Top-likelihood genes

In [ ]:
top_genes = adata.var['fit_likelihood'].sort_values(ascending=False).index
scv.pl.scatter(adata, basis=top_genes[:15], ncols=5, frameon=False)


In [ ]:
var_names = ['lz', 'mthl10']
scv.pl.scatter(adata, var_names, frameon=False)
scv.pl.scatter(adata, x='latent_time', y=var_names, frameon=False)


# Cluster-specific top-likelihood genes

In [ ]:
scv.tl.rank_dynamical_genes(adata, groupby='clusters')
df = scv.get_df(adata, 'rank_dynamical_genes/names')
# df.to_csv("2021-04-13_top_markers.csv")
df.to_csv("2023-02-13_top_markers")

In [ ]:
kwargs = dict(frameon=False, size=10, linewidth=1.5)

# scv.pl.scatter(adata, df['0'][:5], ylabel='0', **kwargs, save = "0.pdf")
# scv.pl.scatter(adata, df['stem cell zone'][:5], ylabel='stem cell zone', **kwargs, save = "scatter_stem cell zone.pdf")
# scv.pl.scatter(adata, df['ureter region PC'][:5], ylabel='ureter region PC', **kwargs, save = "scatter_ureter region PC.pdf")
# scv.pl.scatter(adata, df['lower tubule PC'][:5], ylabel='lower tubule PC', **kwargs, save = "scatter_lower tubule PC.pdf")
# scv.pl.scatter(adata, df['upper tubule PC'][:5], ylabel='upper tubule PC', **kwargs, save = "scatter_upper tubule PC.pdf")
# scv.pl.scatter(adata, df['initial segment PC'][:5], ylabel='initial segment PC', **kwargs, save = "scatter_initial segment PC.pdf")
# scv.pl.scatter(adata, df['stellate cells'][:5], ylabel='stellate cells', **kwargs, save = "scatter_stellate cells.pdf")
# scv.pl.scatter(adata, df['Sox21b cells'][:5], ylabel='Sox21b cells', **kwargs, save = "scatter_Sox21b cells.pdf")

# Velocities in cycling progenitors

In [ ]:
# scv.tl.score_genes_cell_cycle(adata)
# scv.pl.scatter(adata, color_gradients=['S_score', 'G2M_score'], smooth=True, perc=[5, 95])

In [ ]:
scv.pl.velocity(adata, df['0'][:30], ncols=1, add_outline=True, dpi = 300, save = "0.png")
scv.pl.velocity(adata, df['1'][:30], ncols=1, add_outline=True, dpi = 300, save = "1.png")
scv.pl.velocity(adata, df['2'][:30], ncols=1, add_outline=True, dpi = 300, save = "2.png")
scv.pl.velocity(adata, df['3'][:30], ncols=1, add_outline=True, dpi = 300, save = "3.png")
scv.pl.velocity(adata, df['4'][:30], ncols=1, add_outline=True, dpi = 300, save = "4.png")
scv.pl.velocity(adata, df['5'][:30], ncols=1, add_outline=True, dpi = 300, save = "5.png")
scv.pl.velocity(adata, df['6'][:30], ncols=1, add_outline=True, dpi = 300, save = "6.png")
scv.pl.velocity(adata, df['7'][:30], ncols=1, add_outline=True, dpi = 300, save = "7.png")
scv.pl.velocity(adata, df['8'][:30], ncols=1, add_outline=True, dpi = 300, save = "8.png")
scv.pl.velocity(adata, df['9'][:30], ncols=1, add_outline=True, dpi = 300, save = "9.png")
scv.pl.velocity(adata, df['10'][:30],ncols=1, add_outline=True, dpi = 300, save = "10.png")
scv.pl.velocity(adata, df['11'][:30],ncols=1, add_outline=True, dpi = 300, save = "11.png")
scv.pl.velocity(adata, df['12'][:30],ncols=1, add_outline=True, dpi = 300, save = "12.png")



# Speed and coherence

In [ ]:
scv.tl.velocity_confidence(adata)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adata, c=keys, cmap='coolwarm', perc=[5, 95], dpi = 300, figsize = (7, 7), save = "Speed_and_coherence.pdf")

In [ ]:
df = adata.obs.groupby('clusters')[keys].mean().T
df.style.background_gradient(cmap='coolwarm', axis=1)


# Velocity graph and pseudotime

In [ ]:
scv.pl.velocity_graph(adata, threshold=0.5, dpi = 300, figsize = (7, 7), 
                      color = "clusters", palette=pal,
                      save = "velocity_graph.pdf")

In [ ]:
x, y = scv.utils.get_cell_transitions(adata, basis='umap', starting_cell=100)
ax = scv.pl.velocity_graph(adata, c='lightgrey', edge_width=.05, show=False)
ax = scv.pl.scatter(adata, x=x, y=y, s=120, c='ascending', cmap='gnuplot', ax=ax)


In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color='velocity_pseudotime', cmap='gnuplot', dpi = 300, save = "velocity_pseudotime.png")


# PAGA velocity graph

In [ ]:
# PAGA requires to install igraph, if not done yet.
# !pip install python-igraph --upgrade --quiet

In [ ]:
# this is needed due to a current bug - bugfix is coming soon.
adata.uns['neighbors']['distances'] = adata.obsp['distances']
adata.uns['neighbors']['connectivities'] = adata.obsp['connectivities']

scv.tl.paga(adata, groups='clusters')
df = scv.get_df(adata, 'paga/transitions_confidence', precision=2).T
df.style.background_gradient(cmap='Blues').format('{:.2g}')

In [ ]:
scv.pl.paga(adata, basis = "umap", size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5, 
            color = "clusters", palette=pal,
            dpi = 300, figsize = (7, 7), save = "paga.pdf")


In [ ]:
!mkdir data

In [ ]:
# adata.write('data/2020-08-17_MT_stochastic_model.h5ad', compression='gzip')
adata.write('data/2023-02-13_MT_stochastic_model.h5ad', compression='gzip')